# create sp for logging

In [50]:
use [sample];
GO

SELECT DB_NAME() AS db_name;
GO

Commands completed successfully.

(1 row affected)

db_name
-------
sample 
(1 row)

## create tblLog

In [51]:
CREATE OR ALTER PROCEDURE dbo.spCreate_table_log
    @DropIfExists BIT = 0
AS
BEGIN
    IF @DropIfExists = 1
    BEGIN
        DROP TABLE IF EXISTS dbo.tblLog;
        EXEC dbo.spInfo 'DELETED dbo.tblLog'
    END
    
    IF OBJECT_ID('dbo.tblLog', 'U') IS NULL
    BEGIN
        CREATE TABLE dbo.tblLog (
            -- TimeStamp DATETIME DEFAULT CONVERT(NVARCHAR, GETDATE(), 120),
            TimeStamp DATETIME DEFAULT GETDATE(),
            Level NVARCHAR(10),
            Message NVARCHAR(MAX),
            isDisplayed BIT DEFAULT 0
        )
        EXEC dbo.spInfo 'CREATED dbo.tblLog'
    END
END

The module 'spCreate_table_log' depends on the missing object 'dbo.spInfo'. The module will still be created; however, it cannot run successfully until the object exists.
The module 'spCreate_table_log' depends on the missing object 'dbo.spInfo'. The module will still be created; however, it cannot run successfully until the object exists.

In [52]:
EXEC dbo.spCreate_table_log;
SELECT * FROM dbo.tblLog;

(41 rows affected)

TimeStamp               | Level | Message | isDisplayed
------------------------+-------+---------+------------
2026-03-26 16:38:48.320 | INFO  | hello   | 1          
2026-03-26 16:38:48.323 | INFO  | hi      | 1          
2026-03-26 16:38:58.093 | INFO  | hello   | 1          
2026-03-26 16:38:58.097 | INFO  | hi      | 1          
2026-03-26 16:39:45.700 | INFO  | hello   | 1          
2026-03-26 16:39:45.707 | INFO  | hi      | 1          
2026-03-26 16:39:48.827 | INFO  | hello   | 1          
2026-03-26 16:39:48.830 | INFO  | hi      | 1          
2026-03-26 16:39:53.690 | INFO  | hello   | 1          
2026-03-26 16:39:53.693 | INFO  | hi      | 1          
2026-03-26 16:40:41.460 | INFO  | hello   | 1          
2026-03-26 16:40:41.463 | INFO  | hi      | 1          
2026-03-26 16:41:59.173 | INFO  | hello   | 1          
2026-03-26 16:41:59.180 | INFO  | hi      | 1          
2026-03-26 16:42:04.987 | INFO  | hello   | 1          
2026-03-26 16:42:04.993 | INFO  | hi      | 1   

## spLog - base sp for logging

In [53]:
/*
IF OBJECT_ID('spLog', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spLog
    PRINT 'dbo.spLog DELETED'
END
IF OBJECT_ID('spInfo', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spInfo
    PRINT 'dbo.spInfo DELETED'
END
IF OBJECT_ID('spWarn', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spWarn
    PRINT 'dbo.spWarn DELETED'
END
IF OBJECT_ID('spError', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spError
    PRINT 'dbo.spError DELETED'
END
*/
DROP PROCEDURE IF EXISTS 
        dbo.spLog, 
        dbo.spInfo, 
        dbo.spWarn, 
        dbo.spError,
        dbo.spCreate_table_log,
        dbo.spFlush;
PRINT 'Cleanup complete.';

Cleanup complete.

In [54]:
CREATE OR ALTER PROCEDURE dbo.spFLush
AS
BEGIN
    SELECT
        [Timestamp],
        [Level],
        [Message],
        [isDisplayed]
    FROM tblLog
    WHERE isDisplayed = 0;

    UPDATE dbo.tblLog
    SET [isDisplayed] = 1
    WHERE [isDisplayed] = 0;
END

Commands completed successfully.

In [55]:
CREATE OR ALTER PROCEDURE dbo.spLog
    @Level NVARCHAR(10),    -- failing to specify size defaults to 1 !!!
    @Message NVARCHAR(MAX),
    @Flush BIT = 0
AS
BEGIN
    INSERT INTO dbo.tblLog
    (Level, Message)
    SELECT
        -- GETDATE() as [Timestamp],
        @Level as [Level], 
        @Message as [Message];

    IF @Flush = 1
        EXEC dbo.spFlush

    DECLARE @Date NVARCHAR(20) = CONVERT(NVARCHAR, GETDATE(), 120) -- Style 120 is the "Golden Standard"
    SET @Level = UPPER(@Level)
    -- PRINT '@Date: ' + @Date;
    -- PRINT '@Level: ' + @Level;
    DECLARE @Msg NVARCHAR(MAX) = FORMATMESSAGE('%s | %s | %s', 
        @Date,
        @Level, 
        @Message
    );
    PRINT @Msg;
END
GO


Commands completed successfully.

In [56]:
EXEC dbo.spLog 'INFO', 'hello';
EXEC dbo.spLog 'INFO', 'hi', 1;

(1 row affected)
2026-03-26 16:44:26 | INFO | hello
(1 row affected)
(2 rows affected)
(2 rows affected)
2026-03-26 16:44:26 | INFO | hi

Timestamp               | Level | Message | isDisplayed
------------------------+-------+---------+------------
2026-03-26 16:44:26.457 | INFO  | hello   | 0          
2026-03-26 16:44:26.460 | INFO  | hi      | 0          
(2 rows)

## add info, warning, error helpers

In [40]:
CREATE OR ALTER PROCEDURE dbo.spInfo
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'INFO', @Message;
END
GO


Commands completed successfully.

In [41]:
CREATE OR ALTER PROCEDURE dbo.spWarn
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'WARN', @Message;
END
GO


Commands completed successfully.

In [42]:
CREATE OR ALTER PROCEDURE dbo.spError
    @Message NVARCHAR(MAX)
AS
BEGIN
    DECLARE @ErrorMsg NVARCHAR(MAX) = ''
    -- SELECT ERROR_NUMBER(), ERROR_MESSAGE();
    IF ERROR_MESSAGE() IS NOT NULL
        SET @ErrorMsg = FORMATMESSAGE('%s: %i - %s', 
            @Message, 
            ERROR_NUMBER(), 
            ERROR_MESSAGE()
        );
    ELSE
        SET @ErrorMsg = @Message;
    

    EXEC dbo.spLog 'ERROR', @ErrorMsg;
END
GO


Commands completed successfully.

## test sps

In [43]:
SELECT DB_NAME() AS db_name;
GO

PRINT 'start'
EXEC dbo.spLog @Level='INFO', @Message='hello'
EXEC dbo.spInfo @Message='hello'
EXEC dbo.spWarn @Message='hello'
EXEC dbo.spError @Message='hello'
PRINT 'end'
GO


(1 row affected)

db_name
-------
sample 
(1 row)

start
(1 row affected)
2026-03-25 16:59:12 | INFO | hello
(1 row affected)
2026-03-25 16:59:12 | INFO | hello
(1 row affected)
2026-03-25 16:59:12 | WARN | hello
(1 row affected)
2026-03-25 16:59:12 | ERROR | hello
end

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 16:59:12.773 | INFO  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 16:59:12.773 | INFO  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 16:59:12.773 | WARN  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 16:59:12.773 | ERROR | hello  
(1 row)